# Análisis de tweets y modelo de confianza del consumidor

Este notebook carga los datos de tweets y la serie de confianza del consumidor, prepara el dataset mensual, calcula sentimiento con TextBlob y entrena modelos de regresión.

In [ ]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns

from textblob import TextBlob

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from pathlib import Path

sns.set(style='whitegrid')

In [ ]:
# Rutas de los datos
candidate_tweet_paths = [
    Path('gobierno_economia.csv'),
    Path('tweets_gobierno_economia.csv'),
    Path('data/gobierno_economia.csv'),
    Path('data/tweets_gobierno_economia.csv'),
    Path('C:/Users/NADIA/TFG/data/tweets_gobierno_economia.csv'),
]
candidate_icc_paths = [
    Path('evolucion_de_la_confianza_del_consumidor_desde_2004.csv'),
    Path('data/evolucion_de_la_confianza_del_consumidor_desde_2004.csv'),
    Path('icc.csv'),
]

tweets_path = next((p for p in candidate_tweet_paths if p.exists()), None)
icc_path = next((p for p in candidate_icc_paths if p.exists()), None)

if tweets_path is None:
    raise FileNotFoundError('No encontré el CSV de tweets. Revisa candidate_tweet_paths.')
if icc_path is None:
    raise FileNotFoundError('No encontré el CSV de ICC. Revisa candidate_icc_paths.')

print('Tweets cargado desde:', tweets_path)
print('ICC cargado desde:', icc_path)

tweets = pd.read_csv(tweets_path)
icc = pd.read_csv(icc_path, sep=';', encoding='utf-8', engine='python')

print('Tweets:', tweets.shape)
print('ICC:', icc.shape)

display(tweets.head())
display(icc.head())

if 'contenido' not in tweets.columns:
    raise ValueError('No existe la columna "contenido". Columnas disponibles: ' + str(tweets.columns.tolist()))

In [ ]:
# Preparar fechas de los tweets
tweets['fecha_dt'] = pd.to_datetime(
    tweets['fecha'].astype(str).str.replace(' · ', ' ', regex=False),
    format='%b %d, %Y %I:%M %p %Z',
    errors='coerce'
)

if tweets['fecha_dt'].isna().sum() > 0:
    tweets['fecha_dt'] = pd.to_datetime(
        tweets['fecha'].astype(str).str.replace(' · ', ' ', regex=False),
        errors='coerce'
    )

print('Fechas nulas:', tweets['fecha_dt'].isna().sum())
print('Fecha mínima:', tweets['fecha_dt'].min())
print('Fecha máxima:', tweets['fecha_dt'].max())

tweets['mes'] = tweets['fecha_dt'].dt.tz_convert(None).dt.to_period('M').dt.to_timestamp()

display(tweets[['fecha', 'fecha_dt', 'mes']].head())

In [ ]:
# Limpiar texto

def limpiar_texto(texto):
    texto = str(texto)
    texto = re.sub(r'http\S+|www\.\S+', ' ', texto)
    texto = re.sub(r'@\w+', ' ', texto)
    texto = re.sub(r'#', '', texto)
    texto = re.sub(r'\n', ' ', texto)
    texto = re.sub(r'\s+', ' ', texto).strip()
    return texto

tweets['texto_limpio'] = tweets['contenido'].apply(limpiar_texto)

display_columns = [c for c in ['fecha', 'username', 'contenido', 'texto_limpio'] if c in tweets.columns]
if display_columns:
    display(tweets[display_columns].head())
else:
    display(tweets[['contenido', 'texto_limpio']].head())

In [ ]:
# Calcular sentimiento con TextBlob

def calcular_polaridad(texto):
    try:
        return TextBlob(str(texto)).sentiment.polarity
    except Exception:
        return np.nan

tweets['polaridad'] = tweets['texto_limpio'].apply(calcular_polaridad)

tweets['sentimiento_positivo'] = (tweets['polaridad'] > 0.05).astype(int)

tweets['sentimiento_negativo'] = (tweets['polaridad'] < -0.05).astype(int)

display(tweets[['texto_limpio', 'polaridad', 'sentimiento_positivo', 'sentimiento_negativo']].head())

In [ ]:
# Agrupar tweets por mes y crear dataset final
monthly = tweets.groupby('mes').agg(
    tweet_count=('texto_limpio', 'count'),
    avg_polarity=('polaridad', 'mean'),
    positive_share=('sentimiento_positivo', 'mean'),
    negative_share=('sentimiento_negativo', 'mean'),
).reset_index()

possible_date_cols = [c for c in icc.columns if 'fecha' in c.lower() or 'mes' in c.lower() or 'periodo' in c.lower()]
possible_value_cols = [c for c in icc.columns if 'confianza' in c.lower() or 'icc' in c.lower() or 'valor' in c.lower()]

if not possible_date_cols or not possible_value_cols:
    raise ValueError('No encontré columnas de fecha/valor claras en ICC. Columnas: ' + str(list(icc.columns)))

fecha_col = possible_date_cols[0]
valor_col = possible_value_cols[0]
print('Usando fecha ICC:', fecha_col)
print('Usando valor ICC:', valor_col)

icc[fecha_col] = pd.to_datetime(icc[fecha_col], errors='coerce')
icc['mes'] = icc[fecha_col].dt.to_period('M').dt.to_timestamp()

icc_monthly = icc[['mes', valor_col]].rename(columns={valor_col: 'icc_value'})
final = monthly.merge(icc_monthly, on='mes', how='left')
print('Dataset final:', final.shape)
display(final.head(12))

final = final.dropna(subset=['icc_value']).reset_index(drop=True)

features = ['tweet_count', 'avg_polarity', 'positive_share', 'negative_share']
X = final[features]
y = final['icc_value']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

models = {
    'LinearRegression': LinearRegression(),
    'RandomForest': RandomForestRegressor(random_state=42, n_estimators=100),
    'GradientBoosting': GradientBoostingRegressor(random_state=42, n_estimators=100),
}

results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    results.append({
        'model': name,
        'mae': mean_absolute_error(y_test, y_pred),
        'rmse': np.sqrt(mean_squared_error(y_test, y_pred)),
        'r2': r2_score(y_test, y_pred),
    })
    print(f'=== {name} ===')
    print('MAE:', round(results[-1]['mae'], 4))
    print('RMSE:', round(results[-1]['rmse'], 4))
    print('R2:', round(results[-1]['r2'], 4))
    print()

results_df = pd.DataFrame(results)
display(results_df)